# Qdrant dense integration — baseline `D_full_enriched`

Questo notebook trasferisce in Qdrant **la configurazione risultata migliore** nel
notebook di ablation sui 50 film:

- rappresentazione documentale: `D_full_enriched`;
- retriever: dense con `BAAI/bge-m3`;
- risultato osservato: NDCG@10 `0.806716`, Recall@10 `0.793732`,
  Precision@10 `0.500000`.

L'obiettivo non è introdurre un nuovo modello, ma verificare una migrazione
equivalente:

> stesso documento → stesso embedding → stesso filtro → stesso ordinamento top-k.

MCP, LLM, BM25, hybrid e reranker restano deliberatamente fuori da questo notebook.


## Struttura

1. prerequisiti e configurazione centrale;
2. caricamento e validazione del catalogo;
3. costruzione versionata del documento D;
4. embedding BGE-M3 e cache locale controllata;
5. creazione della collection e dei payload index;
6. upsert dei 50 film;
7. query dense con e senza filtri;
8. confronto NumPy ↔ Qdrant;
9. ricalcolo opzionale delle metriche con le annotazioni esistenti.


## 0. Prerequisiti

Dalla cartella radice del pacchetto:

```bash
docker compose up -d
python -m pip install -e ".[bge]"
```

Verificare poi `http://localhost:6333/dashboard`.

Le versioni fissate per questo passaggio sono Qdrant server `1.19.1`,
`qdrant-client 1.19.0` e `FlagEmbedding 1.4.2`.


In [ ]:
# Se si lavora in un ambiente nuovo, eseguire una volta dalla root del progetto:
# %pip install -e ".[bge]"


## 1. Import e individuazione del progetto

Le variabili Hugging Face vengono impostate prima del caricamento effettivo di
`FlagEmbedding`. Il package `movie_rag` contiene le funzioni estratte dal notebook
precedente: il notebook corrente orchestra le chiamate, non le ridefinisce.


In [ ]:
from __future__ import annotations

import importlib.metadata as importlib_metadata
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / "src" / "movie_rag").is_dir():
            return candidate
    raise RuntimeError(
        "Non trovo src/movie_rag. Avviare il notebook dalla cartella del pacchetto "
        "oppure aggiungere manualmente la root del progetto."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from movie_rag import (
    BASELINE,
    BGEM3DenseEncoder,
    DenseMovieRetriever,
    catalog_fingerprint,
    connect_qdrant,
    ensure_dense_collection,
    ensure_payload_indexes,
    evaluate_results,
    index_catalog,
    load_or_encode_documents,
    prepare_catalog,
    validate_query_set,
)
from movie_rag.documents import coerce_point_id

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 100)

print("Project root:", PROJECT_ROOT)
for package in ("pandas", "numpy", "qdrant-client", "FlagEmbedding"):
    try:
        print(f"{package}: {importlib_metadata.version(package)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"{package}: non installato")


Project root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto
pandas: 3.0.5
numpy: 2.5.1
qdrant-client: 1.19.0
FlagEmbedding: 1.4.2


## 2. Configurazione centrale

Modificare qui percorsi e connessione. Le scelte metodologiche della baseline sono
invece centralizzate in `BASELINE` e versionate nel codice.

`RECREATE_COLLECTION=False` evita cancellazioni involontarie. Impostarlo a `True`
elimina e ricrea **soltanto** `COLLECTION_NAME`.


In [ ]:
# Dataset: usato solo se df_test_enriched non è già presente in memoria.
DATASET_PATH = PROJECT_ROOT / "datasets" / "processed" / "dataset_enriched_movielens_tmdb.csv"

# File con i giudizi 0/1/2 del notebook precedente (opzionale).
ANNOTATION_CANDIDATES = [
    PROJECT_ROOT / "artifacts" / "retrieval_v1" / "annotation_pool_50_movies_AA.csv",
    # PROJECT_ROOT / "notebooks" / "annotation_pool_50_movies_AA.csv",
    # PROJECT_ROOT / "annotation_pool_50_movies_AA.csv",
]
ANNOTATION_PATH = next(
    (path for path in ANNOTATION_CANDIDATES if path.exists()),
    ANNOTATION_CANDIDATES[0],
)

# Cache degli embedding: viene riutilizzata solo se ID, fingerprint e configurazione coincidono.
EMBEDDING_CACHE_PATH = (
    PROJECT_ROOT / ".cache" / "embeddings" / "d_full_enriched_bge_m3_v1.npz"
)

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or None
COLLECTION_NAME = BASELINE.collection_name

EXPECTED_N_FILMS = 50
TOP_K = 10
UPSERT_BATCH_SIZE = 64
RECREATE_COLLECTION = False  # Distruttivo per COLLECTION_NAME se impostato a True.
CREATE_PAYLOAD_INDEXES = True

print("Baseline:", BASELINE.name, BASELINE.version)
print("Collection:", COLLECTION_NAME)
print("Qdrant URL:", QDRANT_URL)
print("Dataset fallback:", DATASET_PATH)


Baseline: D_full_enriched_dense v1
Collection: movies_d_full_enriched_dense_v1
Qdrant URL: http://localhost:6333
Dataset fallback: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/dataset_enriched_movielens_tmdb.csv


In [6]:
# Contratto congelato del documento, nell'ordine reale di serializzazione.
baseline_fields = pd.DataFrame(
    BASELINE.document_fields,
    columns=["label_nel_documento", "colonna_dataframe"],
)
display(baseline_fields)


,label_nel_documento,colonna_dataframe
0,Title,canonical_title
1,genres,tmdb_genres
2,Overview,overview
3,Tagline,tagline
4,keywords,tmdb_keyword_text
5,user tags,ml_tags
6,Directors,directors
7,Writers,writers
8,Composers,composers
9,Main cast,cast_top


## 3. Caricamento e validazione del catalogo

Se `df_test_enriched` esiste già nel kernel, viene riutilizzato. In caso contrario
viene letto il CSV indicato sopra. `prepare_catalog` verifica:

- presenza delle colonne richieste da D;
- unicità e validità di `movieId`;
- assenza di documenti vuoti;
- ordine e label esatti dei campi.


In [12]:
if "df_test_enriched" in globals():
    source_df = df_test_enriched.copy()
    print("Uso df_test_enriched già presente in memoria.")
elif DATASET_PATH.exists():
    source_df = pd.read_csv(DATASET_PATH)
    print("Dataset caricato da:", DATASET_PATH)
else:
    raise FileNotFoundError(
        f"Non trovo {DATASET_PATH}. Caricare df_test_enriched oppure modificare DATASET_PATH."
    )

if len(source_df) != EXPECTED_N_FILMS:
    warnings.warn(
        f"La baseline è stata selezionata su {EXPECTED_N_FILMS} film; qui ne risultano {len(source_df)}."
    )

prepared_df = prepare_catalog(source_df)
dataset_fingerprint = catalog_fingerprint(prepared_df)

print("Shape:", prepared_df.shape)
print("Fingerprint catalogo:", dataset_fingerprint)
display(
    prepared_df[
        [BASELINE.id_column, BASELINE.title_column, BASELINE.document_column]
    ].head(3)
)


Dataset caricato da: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/dataset_enriched_movielens_tmdb.csv
Shape: (50, 53)
Fingerprint catalogo: eabb99869fe971b109ef7e23f63ebb67854e8128b5da6de259202d777672287c


,movieId,canonical_title,document__D_full_enriched
0,356,Forrest Gump (1994),"Title: Forrest Gump (1994)\ngenres: Comedy, Drama, Romance\nOverview: A man with a low IQ has accomplished great things in his life and ..."
1,318,"Shawshank Redemption, The (1994)","Title: Shawshank Redemption, The (1994)\ngenres: Drama, Crime\nOverview: Imprisoned in the 1940s for the double murder of his wife and h..."
2,296,Pulp Fiction (1994),"Title: Pulp Fiction (1994)\ngenres: Thriller, Crime, Comedy\nOverview: A burger-loving hit man, his philosophical partner, a drug-addled..."


In [13]:
# Ispezione integrale di un documento: utile per controllare che D sia davvero invariato.
EXAMPLE_POSITION = 0
print(prepared_df.loc[EXAMPLE_POSITION, BASELINE.document_column])


Title: Forrest Gump (1994)
genres: Comedy, Drama, Romance
Overview: A man with a low IQ has accomplished great things in his life and been present during significant historic events—in each case, far exceeding what anyone imagined he could do. But despite all he has achieved, his one true love eludes him.
Tagline: The world will never be the same once you've seen it through the eyes of Forrest Gump.
keywords: new year's eve vietnam war vietnam veteran mentally disabled friendship usa president washington dc, usa post-traumatic stress disorder (ptsd) waitress based on novel or book hippie single parent parent child relationship optimism 1970s drug addiction autism alabama black panther party bus stop family relationships chameleon single mother military anti war protest 1960s simpleton college american football mother son relationship america feelgood dramatic
user tags: Vietnam, bittersweet, bubba gump shrimp, emotional, heartwarming, lieutenant dan, shrimp, stupid is as stupid does, t

## 4. Embedding BGE-M3

`BGEM3DenseEncoder` replica i parametri del notebook precedente:

- output `dense_vecs`;
- `document_max_length=1024`;
- `query_max_length=512`;
- batch `16`;
- normalizzazione L2.

La cache non è una seconda fonte di verità: viene accettata soltanto se coincide il
fingerprint del catalogo, l'ordine degli ID, il modello e la versione documentale.


In [14]:
encoder = BGEM3DenseEncoder(use_fp16=None)

movie_ids = [coerce_point_id(value) for value in prepared_df[BASELINE.id_column]]
documents = prepared_df[BASELINE.document_column].astype(str).tolist()

embedding_artifact = load_or_encode_documents(
    documents=documents,
    movie_ids=movie_ids,
    dataset_fingerprint=dataset_fingerprint,
    encoder=encoder,
    cache_path=EMBEDDING_CACHE_PATH,
)
dense_vectors = embedding_artifact.vectors

assert dense_vectors.shape == (len(prepared_df), BASELINE.embedding_dimension)
norms = np.linalg.norm(dense_vectors, axis=1)
assert np.allclose(norms, 1.0, atol=1e-5)

print("Origine embedding:", embedding_artifact.source)
print("Shape:", dense_vectors.shape)
print("Norma min/max:", float(norms.min()), float(norms.max()))
print("Cache:", EMBEDDING_CACHE_PATH)


Origine embedding: cache
Shape: (50, 1024)
Norma min/max: 0.9999999403953552 1.0000001192092896
Cache: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/.cache/embeddings/d_full_enriched_bge_m3_v1.npz


## 5. Connessione e collection Qdrant

La collection usa un named vector `dense`. Anche se oggi esiste un solo vettore,
il nome esplicito evita una migrazione dello schema se in futuro verrà aggiunta una
rappresentazione sparse o un secondo embedding.

Cosine è coerente con il prodotto scalare sui vettori L2-normalizzati del notebook.


In [7]:
client = connect_qdrant(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout_seconds=30,
)

# È un controllo di raggiungibilità: se fallisce, verificare Docker e QDRANT_URL.
collections_before = client.get_collections()
print("Qdrant raggiungibile.")
print("Collection esistenti:", [item.name for item in collections_before.collections])


Qdrant raggiungibile.
Collection esistenti: ['movies_d_full_enriched_dense_v1', 'test_collection_2']


In [8]:
collection_result = ensure_dense_collection(
    client=client,
    collection_name=COLLECTION_NAME,
    recreate=RECREATE_COLLECTION,
)
print(collection_result)

if CREATE_PAYLOAD_INDEXES:
    indexed_fields = ensure_payload_indexes(client, COLLECTION_NAME)
    print("Payload index creati/confermati:", indexed_fields)


CollectionResult(collection_name='movies_d_full_enriched_dense_v1', created=False, vector_name='dense', vector_size=1024)
Payload index creati/confermati: ['release_year', 'runtime', 'original_language_norm', 'countries_norm', 'genres_norm', 'adult', 'representation_version', 'dataset_fingerprint']


## 6. Upsert dei film

Ogni punto usa `movieId` come ID. Ripetere l'upsert aggiorna lo stesso punto anziché
crearne un duplicato. Se la collection contiene punti estranei al catalogo corrente,
l'indicizzazione si interrompe: usare un nuovo nome o ricreare consapevolmente la
collection, così i risultati non mescolano versioni diverse del dataset.


In [15]:
index_result = index_catalog(
    client=client,
    prepared=prepared_df,
    vectors=dense_vectors,
    collection_name=COLLECTION_NAME,
    batch_size=UPSERT_BATCH_SIZE,
)
print(index_result)

if index_result.collection_points != index_result.dataset_points:
    raise RuntimeError(
        "La collection contiene punti di un'altra versione del catalogo. "
        "Usare un nuovo COLLECTION_NAME o RECREATE_COLLECTION=True."
    )


IndexResult(collection_name='movies_d_full_enriched_dense_v1', indexed_points=50, collection_points=50, dataset_points=50, dataset_fingerprint='eabb99869fe971b109ef7e23f63ebb67854e8128b5da6de259202d777672287c')


In [16]:
# Controllo di un punto: il vettore non viene restituito per evitare output inutilmente grande.
example_id = movie_ids[0]
stored = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[example_id],
    with_payload=True,
    with_vectors=False,
)
assert len(stored) == 1
print("Point ID:", stored[0].id)
display(pd.Series(stored[0].payload))


Point ID: 356


movieId                                                                                                                                                           356
imdbId                                                                                                                                                         109830
tmdbId                                                                                                                                                             13
canonical_title                                                                                                                                   Forrest Gump (1994)
ml_title                                                                                                                                          Forrest Gump (1994)
tmdb_title                                                                                                                                               Forrest Gump
tmdb

## 7. Dense retrieval da Qdrant

`DenseMovieRetriever` è il confine applicativo che verrà riutilizzato più avanti
dall'MCP server. Accetta query, filtri e `top_k`; non dipende dal notebook.

Per i 50 film usiamo `exact=True`, così la verifica riguarda la correttezza della
migrazione e non l'approssimazione ANN/HNSW.


In [17]:
retriever = DenseMovieRetriever(
    client=client,
    encoder=encoder,
    collection_name=COLLECTION_NAME,
)

SIMPLE_QUERY = "un film malinconico sul rapporto tra memoria, identità e tempo"
simple_results = retriever.search(
    SIMPLE_QUERY,
    top_k=TOP_K,
    exact=True,
)
display(
    simple_results[
        ["rank", "movieId", "canonical_title", "score", "release_year", "runtime"]
    ]
)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 42.66it/s]


,rank,movieId,canonical_title,score,release_year,runtime
0,1,4226,Memento (2000),0.531711,2000,113
1,2,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0.530614,1995,129
2,3,2959,Fight Club (1999),0.515896,1999,139
3,4,1270,Back to the Future (1985),0.509704,1985,116
4,5,2571,"Matrix, The (1999)",0.506349,1999,136
5,6,2762,"Sixth Sense, The (1999)",0.489677,1999,107
6,7,589,Terminator 2: Judgment Day (1991),0.483635,1991,137
7,8,780,Independence Day (a.k.a. ID4) (1996),0.483344,1996,145
8,9,590,Dances with Wolves (1990),0.481769,1990,181
9,10,2858,American Beauty (1999),0.474251,1999,122


In [12]:
FILTERED_QUERY = "un film di fantascienza riflessivo sotto le due ore"
FILTERED_QUERY_FILTERS = {
    "max_runtime": 120,
    "include_genres": ["Science Fiction", "Sci-Fi"],
}

filtered_results = retriever.search(
    FILTERED_QUERY,
    filters=FILTERED_QUERY_FILTERS,
    top_k=TOP_K,
    exact=True,
)
columns = [
    "rank",
    "movieId",
    "canonical_title",
    "score",
    "runtime",
    "ml_genres",
    "tmdb_genres",
]
display(filtered_results[[column for column in columns if column in filtered_results]])


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


,rank,movieId,canonical_title,score,runtime,ml_genres,tmdb_genres
0,1,1214,Alien (1979),0.470638,117,Horror|Sci-Fi,"Horror, Science Fiction"
1,2,1270,Back to the Future (1985),0.462818,116,Adventure|Comedy|Sci-Fi,"Adventure, Comedy, Science Fiction"
2,3,1580,Men in Black (a.k.a. MIB) (1997),0.441515,98,Action|Comedy|Sci-Fi,"Action, Adventure, Comedy, Science Fiction"


## 8. Query set e verifica di equivalenza

Riutilizziamo le dieci query definite nel notebook precedente. Per ogni query:

1. calcoliamo una sola volta l'embedding della query;
2. ordiniamo in memoria `dense_vectors @ query_vector`;
3. interroghiamo Qdrant in modalità exact;
4. confrontiamo ID, ordine, overlap e score.

Questa è la condizione di accettazione prima di MCP e LLM.


In [13]:
TEST_QUERIES = pd.DataFrame(
    [
        {
            "query_id": "Q01",
            "category": "semantic_theme_it",
            "query": "un film malinconico sul rapporto tra memoria, identità e tempo",
            "filters": {},
        },
        {
            "query_id": "Q02",
            "category": "semantic_theme_en",
            "query": "a melancholic film about memory, identity and time",
            "filters": {},
        },
        {
            "query_id": "Q03",
            "category": "mood",
            "query": "qualcosa di emozionante, ottimista e commovente",
            "filters": {},
        },
        {
            "query_id": "Q04",
            "category": "named_entity_cast",
            "query": "un film con Tom Hanks",
            "filters": {},
        },
        {
            "query_id": "Q05",
            "category": "named_entity_director",
            "query": "un film diretto da Christopher Nolan",
            "filters": {},
        },
        {
            "query_id": "Q06",
            "category": "collection",
            "query": "un film appartenente a una saga o collection famosa",
            "filters": {},
        },
        {
            "query_id": "Q07",
            "category": "mixed_constraints",
            "query": "un film di fantascienza riflessivo sotto le due ore",
            "filters": {
                "max_runtime": 120,
                "include_genres": ["Science Fiction", "Sci-Fi"],
            },
        },
        {
            "query_id": "Q08",
            "category": "exclusion",
            "query": "un film teso ma non horror",
            "filters": {"exclude_genres": ["Horror"]},
        },
        {
            "query_id": "Q09",
            "category": "year_constraint",
            "query": "un dramma moderno sulle relazioni familiari",
            "filters": {"min_year": 2000},
        },
        {
            "query_id": "Q10",
            "category": "country_language",
            "query": "un film italiano su famiglia e cambiamento sociale",
            "filters": {"countries": ["IT"], "languages": ["it"]},
        },
    ]
)
display(TEST_QUERIES)


,query_id,category,query,filters
0,Q01,semantic_theme_it,"un film malinconico sul rapporto tra memoria, identità e tempo",{}
1,Q02,semantic_theme_en,"a melancholic film about memory, identity and time",{}
2,Q03,mood,"qualcosa di emozionante, ottimista e commovente",{}
3,Q04,named_entity_cast,un film con Tom Hanks,{}
4,Q05,named_entity_director,un film diretto da Christopher Nolan,{}
5,Q06,collection,un film appartenente a una saga o collection famosa,{}
6,Q07,mixed_constraints,un film di fantascienza riflessivo sotto le due ore,"{'max_runtime': 120, 'include_genres': ['Science Fiction', 'Sci-Fi']}"
7,Q08,exclusion,un film teso ma non horror,{'exclude_genres': ['Horror']}
8,Q09,year_constraint,un dramma moderno sulle relazioni familiari,{'min_year': 2000}
9,Q10,country_language,un film italiano su famiglia e cambiamento sociale,"{'countries': ['IT'], 'languages': ['it']}"


In [14]:
equivalence = validate_query_set(
    prepared=prepared_df,
    vectors=dense_vectors,
    retriever=retriever,
    queries=TEST_QUERIES,
    top_k=TOP_K,
)

display(
    equivalence[
        [
            "query_id",
            "local_results",
            "qdrant_results",
            "same_order",
            f"overlap@{TOP_K}",
            "max_score_delta",
        ]
    ]
)

failed = equivalence[
    (~equivalence["same_order"])
    | (equivalence[f"overlap@{TOP_K}"] < 1.0)
    | (equivalence["max_score_delta"].fillna(0) > 1e-5)
]
if not failed.empty:
    display(failed[["query_id", "local_ids", "qdrant_ids", "max_score_delta"]])
    raise AssertionError("Migrazione Qdrant non equivalente: esaminare le query sopra.")

print("✓ Tutte le query soddisfano il criterio di equivalenza NumPy ↔ Qdrant.")


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 37.18it/s]


,query_id,local_results,qdrant_results,same_order,overlap@10,max_score_delta
0,Q01,10,10,True,1.0,6.755219e-08
1,Q02,10,10,True,1.0,1.188879e-07
2,Q03,10,10,True,1.0,8.953468e-08
3,Q04,10,10,True,1.0,5.850494e-08
4,Q05,10,10,True,1.0,5.752533e-08
5,Q06,10,10,True,1.0,5.539856e-08
6,Q07,3,3,True,1.0,6.891479e-08
7,Q08,10,10,True,1.0,6.349274e-08
8,Q09,8,8,True,1.0,6.258896e-08
9,Q10,0,0,True,1.0,NaN


✓ Tutte le query soddisfano il criterio di equivalenza NumPy ↔ Qdrant.


## 9. Ricalcolo opzionale delle metriche

Se `ANNOTATION_PATH` punta al CSV annotato `0/1/2`, ricostruiamo i risultati nel
formato lungo e ricalcoliamo NDCG, Recall e Precision. Non è un nuovo esperimento:
serve a confermare che la migrazione preservi anche l'esito quantitativo.


In [15]:
qdrant_records = []
for query_row in TEST_QUERIES.to_dict("records"):
    results = retriever.search(
        query=query_row["query"],
        filters=query_row["filters"],
        top_k=TOP_K,
        exact=True,
    )
    for result in results.to_dict("records"):
        qdrant_records.append(
            {
                "query_id": query_row["query_id"],
                "query": query_row["query"],
                "category": query_row["category"],
                "experiment": BASELINE.experiment_name,
                "system": "dense_qdrant",
                "rank": int(result["rank"]),
                "movieId": result["movieId"],
                "canonical_title": result.get("canonical_title"),
                "score": float(result["score"]),
            }
        )

qdrant_experiment_results = pd.DataFrame(qdrant_records)
print("Righe risultato Qdrant:", len(qdrant_experiment_results))
display(qdrant_experiment_results.head())


Righe risultato Qdrant: 81


,query_id,query,category,experiment,system,rank,movieId,canonical_title,score
0,Q01,"un film malinconico sul rapporto tra memoria, identità e tempo",semantic_theme_it,D_full_enriched,dense_qdrant,1,4226,Memento (2000),0.531711
1,Q01,"un film malinconico sul rapporto tra memoria, identità e tempo",semantic_theme_it,D_full_enriched,dense_qdrant,2,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0.530614
2,Q01,"un film malinconico sul rapporto tra memoria, identità e tempo",semantic_theme_it,D_full_enriched,dense_qdrant,3,2959,Fight Club (1999),0.515896
3,Q01,"un film malinconico sul rapporto tra memoria, identità e tempo",semantic_theme_it,D_full_enriched,dense_qdrant,4,1270,Back to the Future (1985),0.509704
4,Q01,"un film malinconico sul rapporto tra memoria, identità e tempo",semantic_theme_it,D_full_enriched,dense_qdrant,5,2571,"Matrix, The (1999)",0.506349


In [16]:
if ANNOTATION_PATH.exists():
    annotated_pool = pd.read_csv(ANNOTATION_PATH, sep=";")
    qdrant_metrics = evaluate_results(
        qdrant_experiment_results,
        annotated_pool,
        k=TOP_K,
    )
    display(qdrant_metrics)
    display(
        qdrant_metrics[
            [f"ndcg@{TOP_K}", f"recall@{TOP_K}", f"precision@{TOP_K}"]
        ].mean().to_frame("mean")
    )
else:
    print(
        "Metriche non ricalcolate: file annotazioni non trovato. "
        "Modificare ANNOTATION_PATH se si desidera questo controllo."
    )


,query_id,experiment,system,ndcg@10,recall@10,precision@10,judged_items_for_query
0,Q01,D_full_enriched,dense_qdrant,0.854654,0.700000,0.7,22
1,Q02,D_full_enriched,dense_qdrant,0.784354,0.666667,0.6,22
2,Q03,D_full_enriched,dense_qdrant,0.800368,0.615385,0.8,22
3,Q04,D_full_enriched,dense_qdrant,0.970929,1.000000,0.4,26
4,Q05,D_full_enriched,dense_qdrant,1.000000,1.000000,0.2,27
5,Q06,D_full_enriched,dense_qdrant,0.420064,0.461538,0.6,28
6,Q07,D_full_enriched,dense_qdrant,1.000000,1.000000,0.3,3
7,Q08,D_full_enriched,dense_qdrant,0.552857,0.700000,0.7,23
8,Q09,D_full_enriched,dense_qdrant,0.877215,1.000000,0.2,8


,mean
ndcg@10,0.806716
recall@10,0.793732
precision@10,0.500000


## 10. Esito e prossimo confine

Se la sezione di equivalenza termina senza errori, il retriever dense in Qdrant può
essere considerato la versione infrastrutturale della baseline `D_full_enriched`.

Il confine riutilizzabile per la fase successiva è:

```python
retriever.search(query, filters=..., top_k=...)
retriever.get_movies(movie_ids)
```

Queste funzioni possono essere esposte da un futuro server MCP senza duplicare la
logica di embedding, filtro o accesso a Qdrant.

Prima di procedere con MCP/LLM conviene conservare:

- fingerprint stampato dal notebook;
- nome e versione della collection;
- tabella di equivalenza;
- metriche ricalcolate, se disponibili;
- versioni delle dipendenze.


## Documentazione pertinente

- [Qdrant local quickstart](https://qdrant.tech/documentation/quickstart/):
  avvio Docker, collection, upsert e `query_points`.
- [Qdrant filtering](https://qdrant.tech/documentation/search/filtering/):
  `must`, `must_not`, `MatchAny` e range numerici.
- [Qdrant collections](https://qdrant.tech/documentation/manage-data/collections/):
  vector size, cosine distance e named vectors.
- [qdrant-client](https://pypi.org/project/qdrant-client/): client Python e local mode.
- [BGE-M3](https://huggingface.co/BAAI/bge-m3): generazione di `dense_vecs`.


# APPUNTI README - Movie RAG — baseline dense e integrazione Qdrant

Questo pacchetto porta in Qdrant la configurazione risultata migliore nel notebook
`04_retrieval_ablation_50_movies.ipynb`:

- rappresentazione: `D_full_enriched`;
- retriever: dense `BAAI/bge-m3`;
- vettori: 1024 dimensioni, normalizzati L2;
- distanza Qdrant: cosine;
- metriche medie nel notebook: NDCG@10 `0.806716`, Recall@10 `0.793732`,
  Precision@10 `0.500000`.

È una baseline provvisoria e versionata (`v1`). BM25, hybrid, reranker, MCP e LLM
sono volutamente fuori da questo passaggio: prima si verifica che Qdrant replichi
il ranking dense già misurato.

## Struttura

```text
notebooks/05_qdrant_dense_integration.ipynb  esperimento guidato e validazione
src/movie_rag/config.py                      scelte congelate e nomi di versione
src/movie_rag/documents.py                   documento D, payload e fingerprint
src/movie_rag/embeddings.py                  adapter BGE-M3 e cache validata
src/movie_rag/filters.py                     stessi filtri in Pandas e Qdrant
src/movie_rag/qdrant_store.py                collection, upsert e retriever
src/movie_rag/validation.py                  equivalenza NumPy ↔ Qdrant
src/movie_rag/evaluation.py                  metriche estratte dal notebook
configs/baseline_manifest_v1.json            provenienza e risultati della baseline
tests/                                       test senza download del modello BGE-M3
```

## Avvio rapido

Da questa cartella:

```bash
docker compose up -d
python -m pip install -e ".[bge,test]"
```

Poi aprire `notebooks/05_qdrant_dense_integration.ipynb` ed eseguire le celle in
ordine. Il notebook usa `df_test_enriched` se è già in memoria; altrimenti carica
il CSV configurato nella prima sezione.

Qdrant è esposto solo su `127.0.0.1` e conserva i dati nel volume Docker
`qdrant_storage`. La collection non viene cancellata automaticamente. Il flag
`RECREATE_COLLECTION` è `False`; abilitarlo elimina e ricrea soltanto la collection
indicata.

## Cosa viene salvato

Ogni punto Qdrant usa `movieId` come ID e contiene:

- il named vector `dense`;
- il documento completo `D_full_enriched`;
- titoli, sinossi, persone e altri metadati utili alla futura risposta grounded;
- campi normalizzati per filtri esatti su anno, durata, lingua, paese, genere e
  contenuto adulto;
- versione della rappresentazione, modello e fingerprint del catalogo.

Il reranker, quando verrà reintrodotto, resterà un calcolo a runtime: non produce
un vettore da salvare in Qdrant.

## Criterio di accettazione

Con ricerca `exact=True`, la validazione deve mostrare per le query disponibili:

- `same_order=True`;
- `overlap@10=1.0`;
- differenze di score prossime alla precisione numerica.

Se questo controllo fallisce, non si procede ancora con MCP: si confrontano ordine
del DataFrame, normalizzazione, testo documentale, filtri e versione del modello.

## Nota sul notebook di ablation

Il notebook originale non viene modificato, così gli output che hanno portato alla
scelta della baseline restano un artefatto di provenienza immutato. Durante la review
sono emersi due punti da non copiare nel nuovo codice:

- nella cella degli import, `from __future__ import annotations` compare dopo istruzioni
  eseguibili; in una riesecuzione pulita deve essere la prima istruzione Python;
- la funzione dimostrativa `search_movies` passa al reranker candidati BM25, benché il
  testo della sezione parli di candidati hybrid. La griglia salvata usa invece
  `include_reranker=False`, quindi questo non determina il risultato dense scelto.

Entrambi i punti sono risolti nei moduli del presente pacchetto; il reranker non viene
ancora reintrodotto.

## Riferimenti

- https://qdrant.tech/documentation/quickstart/
- https://qdrant.tech/documentation/search/filtering/
- https://qdrant.tech/documentation/manage-data/collections/
- https://huggingface.co/BAAI/bge-m3
